# Flujo de entrenamiento en Apache Airflow

Este notebook define el flujo de MLOps de BolsaLAB en Airflow: qué etapas tiene,
en qué orden corren y qué regla del protocolo protege cada una.

**Ejecutar este notebook no entrena nada.** Cada celda `%%writefile` solo escribe
un archivo Python en la carpeta `dags/` de la raíz del repositorio. Airflow lee
esa carpeta y es Airflow quien corre las etapas cuando alguien las dispara.

Se generan tres archivos:

| Archivo | Qué es |
|---|---|
| `dags/bolsalab_comun.py` | Rutas y ajustes que comparten los dos flujos |
| `dags/bolsalab_entrenamiento.py` | Flujo 1: de la encuesta al modelo para la app, sin mirar la prueba |
| `dags/bolsalab_prueba_final.py` | Flujo 2: la medición única en el 20 % de prueba, solo a mano |

**Supuestos.**

- Airflow 3 instalado en su propio entorno, separado del proyecto (sus
  dependencias chocan con las de MLflow). Las etapas corren el código del
  proyecto con `uv run`, en el entorno de `uv.lock`.
- La variable de entorno `BOLSALAB_DIR` apunta a la raíz del repositorio, y
  `AIRFLOW__CORE__DAGS_FOLDER` a `$BOLSALAB_DIR/dags`.
- El notebook se abre desde `notebooks/4_mlops/`, así que `../../dags` es la
  carpeta `dags/` de la raíz.

## 1. El mapa de los dos flujos

```
FLUJO 1 · bolsalab_entrenamiento  (a mano o cuando haya datos nuevos; nunca con horario)

  verificar_fuente
        │
  preparar_datos ──► validar_datos            dataset.py   + tests/test_dataset.py
                          │
                 construir_variables ──► validar_variables      features.py + tests/test_features.py
                                              │
                                       comparar_modelos           train.py (validación anidada 5 x 3)
                                              │
                                       compuerta_calidad          reglas del protocolo sobre benchmark.csv
                                              │
                                       exportar_prototipo         train.py --prototype + MLflow
                                        (se salta si el modelo
                                         de models/ ya es el final)
                                         ┌────┴────┐
                              desempeno_grupos   explicacion_shap    groups.py · explain.py


FLUJO 2 · bolsalab_prueba_final  (solo a mano, una sola vez en toda la vida del proyecto)

  confirmar_prueba_unica ──► medir_en_prueba ──► desempeno_grupos
  (falla si la prueba ya       train.py --final     groups.py
   se usó o si nadie            + registro en MLflow
   confirmó)
```

**Por qué son dos flujos.** El protocolo (sección 7) dice que el 20 % de prueba
se mira una sola vez. Si la medición final fuera una etapa del flujo 1, cada
corrida miraría la prueba otra vez. Por eso vive aparte, sin horario y con
dos candados: alguien tiene que confirmarla al dispararla, y el flujo se niega
a correr si `models/crash_reaction_v1.json` ya trae resultados de prueba.

## 2. Las etapas, una por una

| # | Etapa | Comando | Entra | Sale | Regla que protege |
|---|---|---|---|---|---|
| 1 | `verificar_fuente` | revisa que exista la encuesta | `data/external/crash/japan_panic_selling_2020/` | — | Sin el archivo, las pruebas de datos se saltarían en silencio y el flujo seguiría sin validar nada |
| 2 | `preparar_datos` | `python -m bolsalab.dataset` | la encuesta (Excel) | `data/interim/japan_investors.parquet` | — |
| 3 | `validar_datos` | `pytest tests/test_dataset.py` | la encuesta | — | Protocolo, sección 2: si un valor esperado falla, nadie entrena |
| 4 | `construir_variables` | `python -m bolsalab.features` | `data/interim/` | `data/processed/japan_training.parquet` | — |
| 5 | `validar_variables` | `pytest tests/test_features.py` | la tabla de entrenamiento | — | Las 17 variables y `sold` con los valores de `docs/variables.md` |
| 6 | `comparar_modelos` | `python -m bolsalab.modeling.train` | la tabla de entrenamiento (solo el 80 %) | `reports/benchmark.csv` | Protocolo, secciones 4 a 7: misma partición, validación anidada 5 × 3, mismas métricas |
| 7 | `compuerta_calidad` | función de Python sobre `benchmark.csv` | `reports/benchmark.csv` | el ganador (XCom) | Detiene el flujo si la comparación salió incompleta o peor de lo aceptable |
| 8 | `exportar_prototipo` | `python -m bolsalab.modeling.train --prototype` | la tabla de entrenamiento | `models/crash_reaction_v1.*` y una corrida en MLflow | **Se salta** si el modelo de `models/` ya es el final medido en la prueba, para no reemplazarlo |
| 9 | `desempeno_grupos` | `python -m bolsalab.modeling.groups` | la tabla de entrenamiento | `reports/groups.csv` | Revisa que el modelo no sea mucho peor para algún grupo (sexo, edad, ingreso) |
| 10 | `explicacion_shap` | `python -m bolsalab.explain` | el modelo de `models/` | `reports/shap_*.csv` y figuras | Lo que la app muestra como "lo que más pesó" |

**Lo que queda fuera a propósito.** `experiments.py`, `tuning.py` y los notebooks
de mejora del AUROC son exploración: cada uno responde una pregunta y se corre
una vez. No tiene sentido repetirlos en cada corrida del flujo.

## 3. Ajustes compartidos

Rutas del proyecto y la forma de correr un módulo dentro del entorno del
proyecto. Si `uv` no está en el `PATH` del proceso de Airflow (pasa a menudo),
se cambia `UV` por la ruta completa que da `which uv`.

`retries = 0`: el entrenamiento usa semilla fija, así que si una etapa falla,
volver a correrla da el mismo error. Es mejor que falle de una vez y se lea el
log.

In [ ]:
# %%writefile no crea carpetas: primero se asegura de que exista dags/ en la raíz.
from pathlib import Path

DAGS = Path("../../dags")
DAGS.mkdir(exist_ok=True)
print(f"Los flujos se escriben en {DAGS.resolve()}")

In [ ]:
%%writefile ../../dags/bolsalab_comun.py
"""Ajustes compartidos por los flujos de BolsaLAB en Airflow."""

import os
from datetime import datetime, timedelta

# Raíz del repositorio. Se define al arrancar Airflow: export BOLSALAB_DIR=/ruta/a/bolsalab
if "BOLSALAB_DIR" not in os.environ:
    raise RuntimeError("Falta la variable BOLSALAB_DIR con la ruta del repositorio")
REPO = os.environ["BOLSALAB_DIR"]

# uv corre cada etapa en el entorno del proyecto (uv.lock), no en el de Airflow.
UV = os.environ.get("BOLSALAB_UV", "uv")
RUN = f'cd "{REPO}" && {UV} run python -m '
PYTEST = f'cd "{REPO}" && {UV} run pytest -q '

# Python del entorno del proyecto, para las tareas escritas como funciones.
PROJECT_PYTHON = f"{REPO}/.venv/bin/python"

# Variables que reciben las etapas, además de las del sistema.
ENV = {
    "MLFLOW_TRACKING_URI": os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{REPO}/mlflow.db"),
}

DEFAULT_ARGS = {
    "owner": "grupo-10",
    "retries": 0,
    "execution_timeout": timedelta(hours=2),
}

START_DATE = datetime(2026, 11, 1)

# Código de salida con el que una etapa se marca como "saltada" y no como "fallida".
SKIP = 99

# Revisa si el modelo de models/ ya fue medido en la prueba. Sale con 99 si es así.
GUARD_FINAL_MODEL = (
    "import json, sys; "
    "from bolsalab.config import MODELS_DIR, MODEL_ARTIFACT_NAME; "
    "p = MODELS_DIR / f'{MODEL_ARTIFACT_NAME}.json'; "
    "sys.exit(99 if p.exists() and json.loads(p.read_text()).get('test') else 0)"
)

## 4. Flujo 1: `bolsalab_entrenamiento`

Cada etapa del `Makefile` se vuelve una tarea. Casi todas son `BashOperator`
que llaman al mismo módulo que ya existe; la única escrita como función es la
compuerta de calidad.

**La compuerta de calidad** corre con `@task.external_python` dentro del
entorno del proyecto, para poder importar `bolsalab` y `pandas` sin instalarlos
en el entorno de Airflow. Revisa:

1. Que `benchmark.csv` tenga los 7 modelos y ninguna métrica vacía.
2. El ganador con la misma función del protocolo (`choose_winner`).
3. Que el ganador supere al modelo trivial y a la regresión logística, y que su
   pendiente de calibración esté en un rango razonable.

Los umbrales del punto 3 están en `UMBRALES` y son **una propuesta**: el
protocolo todavía no los fija. Hay que acordarlos en equipo y registrarlos en
`docs/protocolo-evaluacion.md` antes de usar el flujo de verdad.

**Por qué `exportar_prototipo` puede saltarse.** `train.py --prototype`
sobrescribe `models/crash_reaction_v1.*`. Hoy ese archivo es el modelo final,
ya medido en la prueba (AUROC 0,670). La etapa primero revisa el `.json`: si
trae resultados de prueba, sale con el código 99 y Airflow la marca como
*saltada* (rosado en la interfaz), no como fallida. Las etapas siguientes
corren igual (`trigger_rule="none_failed"`) sobre el modelo que ya está.

In [ ]:
%%writefile ../../dags/bolsalab_entrenamiento.py
"""Flujo 1 de BolsaLAB: de la encuesta al modelo para la app, sin mirar la prueba.

dataset -> validar -> features -> validar -> comparar modelos -> compuerta de calidad
-> exportar prototipo (si el modelo de models/ no es el final) -> grupos y SHAP.

Nunca corre train.py --final: eso vive en bolsalab_prueba_final.
"""

from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import dag, task

from bolsalab_comun import (
    DEFAULT_ARGS, ENV, GUARD_FINAL_MODEL, PROJECT_PYTHON, PYTEST, REPO, RUN, SKIP, START_DATE, UV,
)

# Propuesta del equipo, pendiente de registrar en el protocolo.
UMBRALES = {
    "auroc_minimo": 0.60,              # por debajo, algo se rompió en los datos o en el código
    "pendiente_calibracion": (0.8, 1.25),
}


def bash(task_id, command, **kwargs):
    return BashOperator(task_id=task_id, bash_command=command, env=ENV, append_env=True,
                        **kwargs)


@dag(
    dag_id="bolsalab_entrenamiento",
    description="De la encuesta japonesa al modelo de la app, sin mirar la prueba",
    schedule=None,          # a mano o cuando haya datos nuevos; los datos de 2020 no cambian solos
    start_date=START_DATE,
    catchup=False,
    default_args=DEFAULT_ARGS,
    tags=["bolsalab", "entrenamiento"],
)
def bolsalab_entrenamiento():

    # 1. La encuesta tiene que estar: si falta, las pruebas de datos se saltarían en silencio.
    verificar_fuente = bash(
        "verificar_fuente",
        f'cd "{REPO}" && {UV} run python -c "'
        "from bolsalab.config import JAPAN_SURVEY_FILE as f; "
        "assert f.exists(), f'Falta la encuesta: {f}'\"",
    )

    # 2-3. Carga y limpieza, y los valores del protocolo (sección 2).
    preparar_datos = bash("preparar_datos", RUN + "bolsalab.dataset")
    validar_datos = bash("validar_datos", PYTEST + "tests/test_dataset.py")

    # 4-5. Las 17 variables y `sold`.
    construir_variables = bash("construir_variables", RUN + "bolsalab.features")
    validar_variables = bash("validar_variables", PYTEST + "tests/test_features.py")

    # 6. Los 7 modelos con validación anidada 5 x 3. No toca la prueba.
    comparar_modelos = bash("comparar_modelos", RUN + "bolsalab.modeling.train")

    # 7. Reglas de calidad sobre reports/benchmark.csv, en el entorno del proyecto.
    @task.external_python(task_id="compuerta_calidad", python=PROJECT_PYTHON,
                          expect_airflow=False)
    def compuerta_calidad(umbrales: dict) -> dict:
        import pandas as pd

        from bolsalab.modeling.train import BENCHMARK_FILE, MODELS, choose_winner

        benchmark = pd.read_csv(BENCHMARK_FILE)
        faltan = set(MODELS) - set(benchmark["model"])
        if faltan:
            raise ValueError(f"benchmark.csv no tiene estos modelos: {sorted(faltan)}")
        if benchmark[["auroc", "auroc_sd", "calibration_slope"]].isna().any().any():
            raise ValueError("benchmark.csv tiene métricas vacías")

        fila = benchmark.set_index("model")
        ganador = choose_winner(benchmark)
        auroc = fila.loc[ganador, "auroc"]
        pendiente = fila.loc[ganador, "calibration_slope"]

        problemas = []
        if auroc < umbrales["auroc_minimo"]:
            problemas.append(f"AUROC {auroc:.3f} < {umbrales['auroc_minimo']}")
        if ganador != "logistic" and auroc <= fila.loc["logistic", "auroc"]:
            problemas.append("el ganador no supera a la regresión logística")
        lo, hi = umbrales["pendiente_calibracion"]
        if not lo <= pendiente <= hi:
            problemas.append(f"pendiente de calibración {pendiente:.2f} fuera de [{lo}, {hi}]")
        if problemas:
            raise ValueError(f"No pasa la compuerta ({ganador}): " + "; ".join(problemas))

        print(f"Pasa la compuerta: {ganador} con AUROC {auroc:.3f} y pendiente {pendiente:.2f}")
        return {"ganador": ganador, "auroc": float(auroc), "pendiente": float(pendiente)}

    calidad = compuerta_calidad(UMBRALES)

    # 8. Exporta para la app y registra en MLflow, salvo que models/ ya tenga el modelo final.
    exportar_prototipo = bash(
        "exportar_prototipo",
        f'cd "{REPO}" && {UV} run python -c "{GUARD_FINAL_MODEL}" '
        f"&& {UV} run python -m bolsalab.modeling.train --prototype",
        skip_on_exit_code=SKIP,
    )

    # 9-10. Revisiones del modelo que quedó en models/ (corren aunque la exportación se salte).
    desempeno_grupos = bash("desempeno_grupos", RUN + "bolsalab.modeling.groups",
                            trigger_rule="none_failed")
    explicacion_shap = bash("explicacion_shap", RUN + "bolsalab.explain",
                            trigger_rule="none_failed")

    (
        verificar_fuente
        >> preparar_datos >> validar_datos
        >> construir_variables >> validar_variables
        >> comparar_modelos >> calidad
        >> exportar_prototipo >> [desempeno_grupos, explicacion_shap]
    )


bolsalab_entrenamiento()

## 5. Flujo 2: `bolsalab_prueba_final`

La medición en el 20 % de prueba, separada y con dos candados:

1. **Confirmación al dispararla.** El flujo tiene un parámetro
   `confirmo_prueba_unica` que arranca en `false`. En la interfaz, al pulsar
   *Trigger*, Airflow muestra el formulario y alguien tiene que marcarlo. Si
   no, la primera etapa falla y nada se mide.
2. **Una sola vez.** Si `models/crash_reaction_v1.json` ya trae resultados de
   prueba, la primera etapa falla. Hoy ya los trae (el `--final` se corrió en
   `main`), así que este flujo queda documentado pero **bloqueado**: solo
   serviría con datos nuevos y un protocolo nuevo, por ejemplo una versión 2 del
   modelo.

`train.py --final` registra solo el modelo en MLflow como una versión nueva del
modelo `bolsalab-reaccion-caidas` y escribe `reports/groups_test.csv`.

In [ ]:
%%writefile ../../dags/bolsalab_prueba_final.py
"""Flujo 2 de BolsaLAB: la medición única en el 20 % de prueba.

Solo a mano. Falla si nadie confirmó la medición o si el modelo de models/ ya
fue medido en la prueba (protocolo, sección 7: la prueba se mira una sola vez).
"""

from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param, dag

from bolsalab_comun import DEFAULT_ARGS, ENV, GUARD_FINAL_MODEL, REPO, RUN, START_DATE, UV


def bash(task_id, command, **kwargs):
    return BashOperator(task_id=task_id, bash_command=command, env=ENV, append_env=True,
                        **kwargs)


@dag(
    dag_id="bolsalab_prueba_final",
    description="Medición única del ganador en el 20 % de prueba (solo a mano)",
    schedule=None,
    start_date=START_DATE,
    catchup=False,
    default_args=DEFAULT_ARGS,
    max_active_runs=1,
    params={
        "confirmo_prueba_unica": Param(
            False, type="boolean",
            description="Marcar solo si el equipo acordó medir en la prueba y nunca se ha hecho",
        ),
    },
    tags=["bolsalab", "prueba-final"],
)
def bolsalab_prueba_final():

    # Candado 1: alguien confirmó. Candado 2: la prueba no se ha usado.
    # El guardián sale con 99 si ya se usó; aquí eso cuenta como falla, no como salto.
    confirmar_prueba_unica = bash(
        "confirmar_prueba_unica",
        '[ "{{ params.confirmo_prueba_unica }}" = "True" ] '
        '|| { echo "Falta confirmar confirmo_prueba_unica al disparar el flujo"; exit 1; }; '
        f'cd "{REPO}" && {UV} run python -c "{GUARD_FINAL_MODEL}" '
        '|| { echo "El modelo de models/ ya fue medido en la prueba. No se repite."; exit 1; }',
        skip_on_exit_code=None,
    )

    # Entrena al ganador, lo mide en la prueba, lo guarda en models/ y lo registra en MLflow.
    medir_en_prueba = bash("medir_en_prueba", RUN + "bolsalab.modeling.train --final")

    desempeno_grupos = bash("desempeno_grupos", RUN + "bolsalab.modeling.groups")

    confirmar_prueba_unica >> medir_en_prueba >> desempeno_grupos


bolsalab_prueba_final()

## 6. Cómo ponerlo a andar

Después de correr las celdas de arriba (solo escriben los tres archivos):

```bash
# En la terminal donde se arranca Airflow
export BOLSALAB_DIR=/ruta/a/bolsalab
export AIRFLOW__CORE__DAGS_FOLDER=$BOLSALAB_DIR/dags
export AIRFLOW__CORE__LOAD_EXAMPLES=False
export BOLSALAB_UV=$(which uv)          # por si Airflow no hereda el PATH

cd $BOLSALAB_DIR && uv sync             # crea .venv, que usa la compuerta de calidad
airflow standalone                       # interfaz en http://localhost:8080
```

Para revisar sin correr nada:

```bash
airflow dags list                        # deben aparecer los dos flujos
airflow dags list-import-errors          # errores de sintaxis o de imports en dags/
airflow tasks list bolsalab_entrenamiento --tree
```

Para probar una sola etapa sin la interfaz (esta sí corre el código):

```bash
airflow tasks test bolsalab_entrenamiento validar_datos 2026-11-20
```

## 7. Lo que falta decidir en equipo

| Decisión | Por qué importa |
|---|---|
| Los umbrales de `UMBRALES` | Hoy son una propuesta; el protocolo debe fijarlos antes de que la compuerta bloquee algo de verdad |
| Si `dags/` va en el repositorio | Recomendado: el flujo se revisa en los PR como el resto del código |
| MLflow compartido | Con `sqlite:///mlflow.db` cada integrante tiene su propio registro. Un servidor común (fase 2 de la ruta) deja ver las corridas de todos |
| Cuándo se dispara el flujo 1 | Los datos de 2020 no cambian; tiene sentido con datos nuevos (NFCS, una encuesta piloto en Colombia) o con cambios en el código del modelo |